# PMOF action timelines per person

This notebook plots the temporal action sequence of every annotated person in every PMOF recording. Each bar represents one `(recording, track ID)` pair; stacked segments run from the first observed frame at the bottom to the last observed frame at the top.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch

# Paths come from path_config.yaml; edit that file's current_workstation to switch machines.
from path_config import PMOF_CODE_DIR, DATA_BASE_DIR

# Set to None to analyze all recordings, or provide IDs such as ["rec1", "rec7", "rec12"].
SELECTED_RECORD_IDS = [f"rec{i}" for i in range(1,31)]

# When True, show seated_ground annotations as lying in the timeline and legend.
MAP_SEATED_GROUND_TO_LYING = False

if not PMOF_CODE_DIR.is_dir():
    raise FileNotFoundError(f"PMOF code directory not found: {PMOF_CODE_DIR}")
if not (DATA_BASE_DIR / "images").is_dir() or not (DATA_BASE_DIR / "annotations").is_dir():
    raise FileNotFoundError(
        "DATA_BASE_DIR must contain both 'images' and 'annotations' directories. "
        f"Current value: {DATA_BASE_DIR}"
    )

sys.path.insert(0, str(PMOF_CODE_DIR))

from src.data import list_record_ids, read_annotation, recid_to_annpath, recordid_to_imageids
from src.visualization import VIZ_PARAMS

ACTION_COLORS = VIZ_PARAMS["gt_bbox_colors"]
ACTION_ORDER = ["seated", "seated_ground", "standing", "lying"]

In [ ]:
def frame_number(image_id: str) -> int:
    """Extract the numeric frame index from an ID such as ``rec12_000345``."""
    return int(image_id.rsplit("_", maxsplit=1)[1])


def selected_record_ids(data_base_dir: Path, requested_ids: list[str] | None) -> list[str]:
    """Return all PMOF records or a validated user-selected subset."""
    available_ids = list_record_ids(data_base_dir)
    if requested_ids is None:
        return available_ids

    unknown_ids = sorted(set(requested_ids) - set(available_ids))
    if unknown_ids:
        raise ValueError(
            "Selected record IDs are not present in this dataset: "
            + ", ".join(unknown_ids)
        )
    return [record_id for record_id in available_ids if record_id in requested_ids]


def normalized_action(action: str) -> str:
    """Apply the optional display mapping for action annotations."""
    if MAP_SEATED_GROUND_TO_LYING and action == "seated_ground":
        return "lying"
    return action


def load_person_actions(data_base_dir: Path, record_ids: list[str]) -> pd.DataFrame:
    """Read every annotated person action with PMOF's dataset and annotation helpers."""
    action_rows = []

    for record_id in record_ids:
        annotation_path = recid_to_annpath(record_id, data_base_dir)
        for image_id in recordid_to_imageids(record_id, data_base_dir):
            for annotation in read_annotation(annotation_path, image_id):
                if annotation.category_name != "person" or annotation.action is None:
                    continue
                action_rows.append(
                    {
                        "record_id": record_id,
                        "frame": frame_number(image_id),
                        "track_id": annotation.track_id,
                        "action": normalized_action(annotation.action),
                        "occluded": bool(annotation.occluded),
                    }
                )

    frame_actions = pd.DataFrame(
        action_rows, columns=["record_id", "frame", "track_id", "action", "occluded"]
    )
    if frame_actions.empty:
        raise ValueError("No person annotations with an action attribute were found.")

    unsupported_actions = sorted(set(frame_actions["action"]) - set(ACTION_ORDER))
    if unsupported_actions:
        raise ValueError(
            "Expected only PMOF actions "
            f"{ACTION_ORDER}; found: {', '.join(unsupported_actions)}"
        )
    if frame_actions.duplicated(["record_id", "frame", "track_id"]).any():
        raise ValueError("A person has multiple action annotations in the same frame.")

    return frame_actions.sort_values(["record_id", "track_id", "frame"]).reset_index(drop=True)


record_order = selected_record_ids(DATA_BASE_DIR, SELECTED_RECORD_IDS)
frame_actions = load_person_actions(DATA_BASE_DIR, record_order)
frame_actions.head()

In [ ]:
def consecutive_action_runs(person_frames: pd.DataFrame) -> list[dict]:
    """Collapse consecutive equal (action, occluded) pairs while retaining chronological order."""
    runs = []
    current_key = None
    run_start = None
    run_end = None
    frame_count = 0

    for row in person_frames.sort_values("frame").itertuples(index=False):
        key = (row.action, row.occluded)
        if key != current_key:
            if current_key is not None:
                runs.append(
                    {
                        "action": current_key[0],
                        "occluded": current_key[1],
                        "start_frame": run_start,
                        "end_frame": run_end,
                        "frame_count": frame_count,
                    }
                )
            current_key = key
            run_start = row.frame
            frame_count = 0
        run_end = row.frame
        frame_count += 1

    if current_key is not None:
        runs.append(
            {
                "action": current_key[0],
                "occluded": current_key[1],
                "start_frame": run_start,
                "end_frame": run_end,
                "frame_count": frame_count,
            }
        )
    return runs


run_rows = []
for (record_id, track_id), person_frames in frame_actions.groupby(["record_id", "track_id"], sort=True):
    for run in consecutive_action_runs(person_frames):
        run_rows.append({"record_id": record_id, "track_id": track_id, **run})

action_runs = pd.DataFrame(run_rows)
action_runs.head(10)

In [ ]:
record_tracks = {
    record_id: sorted(
        frame_actions.loc[frame_actions["record_id"] == record_id, "track_id"].unique()
    )
    for record_id in record_order
}

record_gap = 1.5
OCCLUDED_COLOR = "grey"
track_order = []
track_positions = []
record_centers = {}
record_top_heights = {}
next_position = 0.0

for record_id in record_order:
    track_ids = record_tracks[record_id]
    positions = list(range(int(next_position), int(next_position) + len(track_ids)))
    track_order.extend((record_id, track_id) for track_id in track_ids)
    track_positions.extend(positions)
    record_centers[record_id] = (positions[0] + positions[-1]) / 2
    next_position = positions[-1] + 1 + record_gap

# shared across both timeline plots so they're directly comparable
TIMELINE_FIGSIZE = (max(12, 0.35 * len(track_order)), 6)

fig, ax = plt.subplots(figsize=TIMELINE_FIGSIZE, dpi=150)

for position, (record_id, track_id) in zip(track_positions, track_order):
    person_runs = action_runs.loc[
        (action_runs["record_id"] == record_id)
        & (action_runs["track_id"] == track_id)
    ].sort_values("start_frame")
    bottom = 0

    for run in person_runs.itertuples(index=False):
        bar_color = OCCLUDED_COLOR if run.occluded else ACTION_COLORS[run.action]
        ax.bar(
            position,
            run.frame_count,
            bottom=bottom,
            width=0.85,
            color=bar_color,
            edgecolor="none",
            antialiased=False,
        )
        bottom += run.frame_count

    record_top_heights[record_id] = max(record_top_heights.get(record_id, 0), bottom)

observed_actions = set(action_runs["action"])
legend_handles = [
    Patch(facecolor=ACTION_COLORS[action], label=action.replace("_", " "))
    for action in ACTION_ORDER
    if action in observed_actions
]
if action_runs["occluded"].any():
    legend_handles.append(Patch(facecolor=OCCLUDED_COLOR, label="occluded"))
ax.legend(handles=legend_handles, ncols=min(4, len(legend_handles)))
ax.set_xticks(track_positions)
ax.set_xticklabels([f"ID {track_id}" for _, track_id in track_order], rotation=90)
ax.set_xlabel("Person Track ID")
ax.set_ylabel("Annotated Frames per Person")
ax.set_title("PMOF Action Timelines by Person")
ax.set_xlim(track_positions[0] - 1.2, track_positions[-1] + 1.2)
ax.grid(axis="y", linewidth=0.4, alpha=0.35)
ax.set_axisbelow(True)

for record_id, center in record_centers.items():
    ax.annotate(
        record_id,
        xy=(center, record_top_heights[record_id]-20),
        xycoords="data",
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontweight="bold",
        clip_on=False,
    )

plt.show()
fig.tight_layout()

In [ ]:
ALERT_COLORS = {False: "tab:green", True: "tab:pink"}


def frame_alert_status(record_frame_actions: pd.DataFrame) -> pd.DataFrame:
    """Return one row per (record_id, frame): alert=True if any person there isn't seated."""
    frame_alert = (
        record_frame_actions.groupby(["record_id", "frame"])["action"]
        .apply(lambda actions: bool((actions != "seated").any()))
        .reset_index(name="alert")
    )
    return frame_alert.sort_values(["record_id", "frame"]).reset_index(drop=True)


def consecutive_alert_runs(record_frames: pd.DataFrame) -> list[dict]:
    """Collapse consecutive equal alert values while retaining chronological order."""
    runs = []
    current_alert = None
    run_start = None
    run_end = None
    frame_count = 0

    for row in record_frames.sort_values("frame").itertuples(index=False):
        if row.alert != current_alert:
            if current_alert is not None:
                runs.append(
                    {
                        "alert": current_alert,
                        "start_frame": run_start,
                        "end_frame": run_end,
                        "frame_count": frame_count,
                    }
                )
            current_alert = row.alert
            run_start = row.frame
            frame_count = 0
        run_end = row.frame
        frame_count += 1

    if current_alert is not None:
        runs.append(
            {
                "alert": current_alert,
                "start_frame": run_start,
                "end_frame": run_end,
                "frame_count": frame_count,
            }
        )
    return runs


frame_alert = frame_alert_status(frame_actions)

alert_run_rows = []
for record_id, record_frames in frame_alert.groupby("record_id", sort=True):
    for run in consecutive_alert_runs(record_frames):
        alert_run_rows.append({"record_id": record_id, **run})

alert_runs = pd.DataFrame(alert_run_rows)
alert_runs.head(10)


In [ ]:
record_positions = list(range(len(record_order)))

# shared across both timeline plots so they're directly comparable
TIMELINE_FIGSIZE = (max(12, 0.35 * len(track_order)), 6)

fig, ax = plt.subplots(figsize=TIMELINE_FIGSIZE, dpi=150)
for position, record_id in zip(record_positions, record_order):
    record_runs = alert_runs.loc[alert_runs["record_id"] == record_id].sort_values("start_frame")
    bottom = 0

    for run in record_runs.itertuples(index=False):
        ax.bar(
            position,
            run.frame_count,
            bottom=bottom,
            width=0.7,
            color=ALERT_COLORS[run.alert],
            edgecolor="none",
            antialiased=False,
        )
        bottom += run.frame_count

    ax.annotate(
        record_id,
        xy=(position, bottom-20),
        xycoords="data",
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontweight="bold",
        clip_on=False,
    )

legend_handles = [
    Patch(facecolor=ALERT_COLORS[False], label="no alert (all seated)"),
    Patch(facecolor=ALERT_COLORS[True], label="alert (not all seated)"),
]
ax.legend(handles=legend_handles)
ax.set_xticks(record_positions)
ax.set_xticklabels(record_order, rotation=0)
ax.set_xlabel("Recording")
ax.set_ylabel("Annotated Frames")
ax.set_title("PMOF Alert Timeline by Recording")
ax.set_xlim(record_positions[0] - 1.2, record_positions[-1] + 1.2)
ax.grid(axis="y", linewidth=0.4, alpha=0.35)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()


In [ ]:
# Combined per-person and per-recording alert timeline: shared x-axis aligned by recording (per-track axis dropped).

In [ ]:
record_widths = {
    record_id: max(len(track_ids) * 0.85, 0.5)
    for record_id, track_ids in record_tracks.items()
}

fig, (ax_actions, ax_alert) = plt.subplots(
    2, 1, figsize=(TIMELINE_FIGSIZE[0], TIMELINE_FIGSIZE[1] * 2),
    sharex=True, dpi=150, gridspec_kw={"height_ratios": [1, 1]},
)

# top: per-person action timeline (same bars as the standalone plot above)
for position, (record_id, track_id) in zip(track_positions, track_order):
    person_runs = action_runs.loc[
        (action_runs["record_id"] == record_id)
        & (action_runs["track_id"] == track_id)
    ].sort_values("start_frame")
    bottom = 0

    for run in person_runs.itertuples(index=False):
        bar_color = OCCLUDED_COLOR if run.occluded else ACTION_COLORS[run.action]
        ax_actions.bar(
            position,
            run.frame_count,
            bottom=bottom,
            width=0.85,
            color=bar_color,
            edgecolor="none",
            antialiased=False,
        )
        bottom += run.frame_count

observed_actions = set(action_runs["action"])
action_legend_handles = [
    Patch(facecolor=ACTION_COLORS[action], label=action.replace("_", " "))
    for action in ACTION_ORDER
    if action in observed_actions
]
if action_runs["occluded"].any():
    action_legend_handles.append(Patch(facecolor=OCCLUDED_COLOR, label="occluded"))
ax_actions.legend(handles=action_legend_handles, ncols=min(4, len(action_legend_handles)))
ax_actions.set_ylabel("Annotated Frames per Person")
ax_actions.set_title("PMOF Action Timelines by Person")
ax_actions.grid(axis="y", linewidth=0.4, alpha=0.35)
ax_actions.set_axisbelow(True)

# bottom: per-recording alert timeline, each bar centered under its recording's tracks
for record_id in record_order:
    position = record_centers[record_id]
    record_runs = alert_runs.loc[alert_runs["record_id"] == record_id].sort_values("start_frame")
    bottom = 0

    for run in record_runs.itertuples(index=False):
        ax_alert.bar(
            position,
            run.frame_count,
            bottom=bottom,
            width=record_widths[record_id],
            color=ALERT_COLORS[run.alert],
            edgecolor="none",
            antialiased=False,
        )
        bottom += run.frame_count

alert_legend_handles = [
    Patch(facecolor=ALERT_COLORS[False], label="no alert (all seated)"),
    Patch(facecolor=ALERT_COLORS[True], label="alert (not all seated)"),
]
ax_alert.legend(handles=alert_legend_handles)
ax_alert.set_ylabel("Annotated Frames")
ax_alert.set_title("PMOF Alert Timeline by Recording")
ax_alert.grid(axis="y", linewidth=0.4, alpha=0.35)
ax_alert.set_axisbelow(True)

# shared x-axis: ticks show only the recording id, the per-track axis is dropped
ax_alert.set_xticks(list(record_centers.values()))
ax_alert.set_xticklabels(list(record_centers.keys()), rotation=0)
ax_alert.set_xlabel("Recording")
ax_alert.set_xlim(track_positions[0] - 1.2, track_positions[-1] + 1.2)

for ax in (ax_actions, ax_alert):
    ax.label_outer()

fig.tight_layout()
plt.savefig("pmof_action_alert_timelines_combined.png", dpi=300)
plt.show()


In [ ]:
def plot_action_alert_pies(record_ids=None):
    """Pie charts of action-label and alert/no-alert distributions, optionally for a record subset."""
    record_ids = record_order if record_ids is None else selected_record_ids(DATA_BASE_DIR, record_ids)

    action_subset = frame_actions.loc[frame_actions["record_id"].isin(record_ids)]
    alert_subset = frame_alert.loc[frame_alert["record_id"].isin(record_ids)]
    if action_subset.empty or alert_subset.empty:
        print(f"No data found for record ids: {record_ids}")
        return

    action_counts = action_subset["action"].value_counts().reindex(ACTION_ORDER).dropna()
    alert_counts = alert_subset["alert"].value_counts()

    fig, (ax_action, ax_alert_pie) = plt.subplots(1, 2, figsize=(10, 5), dpi=150)

    ax_action.pie(
        action_counts,
        labels=[action.replace("_", " ") for action in action_counts.index],
        colors=[ACTION_COLORS[action] for action in action_counts.index],
        autopct="%1.1f%%",
        startangle=90,
    )
    ax_action.set_title("Action Label Distribution")

    ax_alert_pie.pie(
        alert_counts,
        labels=["alert" if is_alert else "no alert" for is_alert in alert_counts.index],
        colors=[ALERT_COLORS[is_alert] for is_alert in alert_counts.index],
        autopct="%1.1f%%",
        startangle=90,
    )
    ax_alert_pie.set_title("Alert / No-Alert Distribution")

    fig.suptitle(f"Records: {', '.join(record_ids)}")
    fig.tight_layout()
    plt.show()


# pass a list of record ids, e.g. plot_action_alert_pies(["rec1", "rec7"]), to inspect a subset
plot_action_alert_pies(['rec4', 'rec22', 'rec25', 'rec29', 'rec30'])


In [ ]:
plot_action_alert_pies(['rec27', 'rec28'])